In [ ]:
# Célula 1 — Imports + leitura dos dados
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

ARQUIVO_DADOS = "outputs_csv/dados_modelagem_ndvi.csv"

if ARQUIVO_DADOS.endswith(".xlsx"):
    df = pd.read_excel(ARQUIVO_DADOS)
else:
    df = pd.read_csv(ARQUIVO_DADOS)

df.head()


In [ ]:
# Célula 2 — Limpeza/preparação
df = df.rename(columns={"NDVI t-1": "NDVI_t_1", "NDVI t": "NDVI_t", "vento ": "vento"})

colunas_numericas = ["DAS_atual", "NDVI_t_1", "NDVI_t", "precipitacao_ac", "temperatura", "radiacao", "umidade", "vento"]

for coluna in colunas_numericas:
    df[coluna] = pd.to_numeric(df[coluna], errors="coerce")

df = df.dropna(subset=["NDVI_t"]).copy()
df_modelo = df.dropna(subset=["NDVI_t_1"]).copy()
df_modelo = df_modelo.sort_values(["safra", "DAS_atual"]).reset_index(drop=True)

df_modelo.info()
df_modelo.head()


In [ ]:
# Célula 3 — Separação treino/teste
TEST_SAFRA = "2023-2024"
TRAIN_SAFRAS = [safra for safra in df_modelo["safra"].unique() if safra != TEST_SAFRA]

df_treino = df_modelo[df_modelo["safra"].isin(TRAIN_SAFRAS)].copy()
df_teste = df_modelo[df_modelo["safra"] == TEST_SAFRA].copy()

resultados = {}
previsoes = {}
modelos = {}

def rodar_modelo(nome, features):
    X_train = df_treino[features]
    y_train = df_treino["NDVI_t"]
    X_test = df_teste[features]
    y_test = df_teste["NDVI_t"]

    modelo = RandomForestRegressor(n_estimators=300, random_state=42)
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)

    resultados[nome] = {
        "Modelo": nome,
        "MAE": mean_absolute_error(y_test, y_pred),
        "RMSE": mean_squared_error(y_test, y_pred, squared=False),
        "R2": r2_score(y_test, y_pred),
    }

    previsoes[nome] = y_pred
    modelos[nome] = modelo
    return pd.DataFrame([resultados[nome]])

print("Safras de treino:", TRAIN_SAFRAS)
print("Safra de teste:", TEST_SAFRA)
print("Linhas treino:", len(df_treino))
print("Linhas teste:", len(df_teste))
df_teste[["safra", "DAS_atual", "NDVI_t_1", "NDVI_t"]]


In [ ]:
# Célula 4 — Modelo base DAS + NDVI_t-1 → NDVI_t
resultado_base = rodar_modelo("Base", ["DAS_atual", "NDVI_t_1"])
resultado_base


In [ ]:
# Célula 5 — Base + precipitação DAS + NDVI_t-1 + precipitação → NDVI_t
resultado_precipitacao = rodar_modelo("Base + precipitação", ["DAS_atual", "NDVI_t_1", "precipitacao_ac"])
resultado_precipitacao


In [ ]:
# Célula 6 — Base + temperatura DAS + NDVI_t-1 + temperatura → NDVI_t
resultado_temperatura = rodar_modelo("Base + temperatura", ["DAS_atual", "NDVI_t_1", "temperatura"])
resultado_temperatura


In [ ]:
# Célula 7 — Base + radiação DAS + NDVI_t-1 + radiação → NDVI_t
resultado_radiacao = rodar_modelo("Base + radiação", ["DAS_atual", "NDVI_t_1", "radiacao"])
resultado_radiacao


In [ ]:
# Célula 8 — Base + umidade DAS + NDVI_t-1 + umidade → NDVI_t
resultado_umidade = rodar_modelo("Base + umidade", ["DAS_atual", "NDVI_t_1", "umidade"])
resultado_umidade


In [ ]:
# Célula 9 — Base + vento DAS + NDVI_t-1 + vento → NDVI_t
resultado_vento = rodar_modelo("Base + vento", ["DAS_atual", "NDVI_t_1", "vento"])
resultado_vento


In [ ]:
# Célula 10 — Todas as variáveis climáticas
resultado_completo = rodar_modelo("Todas as variáveis climáticas", ["DAS_atual", "NDVI_t_1", "precipitacao_ac", "temperatura", "radiacao", "umidade", "vento"])
resultado_completo


In [ ]:
# Célula 11 — Comparação final
comparacao_final = pd.DataFrame(resultados.values()).sort_values("MAE").reset_index(drop=True)
comparacao_final


In [ ]:
# Célula 12 — Gráfico real × previsto
plt.figure(figsize=(12, 6))
plt.plot(df_teste["DAS_atual"], df_teste["NDVI_t"], marker="o", linewidth=3, color="black", label="Real")

for nome, y_pred in previsoes.items():
    plt.plot(df_teste["DAS_atual"], y_pred, marker="o", linewidth=2, label=nome)

plt.title(f"Safra de teste: {TEST_SAFRA} | Curva real × prevista")
plt.xlabel("DAS")
plt.ylabel("NDVI")
plt.grid(True, linestyle=":", alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Célula 13 — Importância das variáveis
variaveis_modelo_completo = ["DAS_atual", "NDVI_t_1", "precipitacao_ac", "temperatura", "radiacao", "umidade", "vento"]

importancias = pd.DataFrame({
    "Variavel": variaveis_modelo_completo,
    "Importancia": modelos["Todas as variáveis climáticas"].feature_importances_
}).sort_values("Importancia", ascending=True)

importancias

plt.figure(figsize=(8, 5))
plt.barh(importancias["Variavel"], importancias["Importancia"], color="forestgreen")
plt.title("Importância das variáveis no Random Forest")
plt.xlabel("Importância")
plt.tight_layout()
plt.show()
